# RepLens — Video review and annotation

Use **Run → Run All Cells** to execute setup, helpers and batch processing. Wait for the progress table; the first successful video then appears under **3. Watch and review**. MediaPipe assists annotation only. Clear video outputs before sharing.

## 1. Setup

Run the setup and collapsed helper cells once. Add videos to `training/data/videos/`. FFmpeg is required for playback. Review files stay under ignored `training/data/`.

In [ ]:
from pathlib import Path
from dataclasses import dataclass
from tempfile import TemporaryDirectory
from types import CodeType
from urllib.request import urlretrieve
from importlib.metadata import version
import csv
import hashlib
import json
import re
import shutil
import subprocess
import time

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.signal import find_peaks, savgol_filter
from IPython.display import Video, display

ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents]
             if (path / ".git").is_dir() and (path / "training" / "notebooks").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open the notebook from the RepLens project folder.")
DATA_DIR = ROOT / "training" / "data"
VIDEO_DIR = DATA_DIR / "videos"
REVIEW_DIR = DATA_DIR / "review"
MANIFEST_PATH = DATA_DIR / "video_manifest.csv"
EXTENSIONS = {".mp4", ".mov", ".avi", ".webm", ".mkv", ".m4v"}
FIELDS = ["video_id", "relative_path", "participant_id", "source_recording_id",
          "source_url", "reuse_status", "reuse_evidence", "exercise",
          "complete_uncut_set", "stop_reason", "endpoint_evidence",
          "label_status", "quality_status", "review_notes"]
VIDEO_DIR.mkdir(parents=True, exist_ok=True)
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
if not MANIFEST_PATH.exists():
    with MANIFEST_PATH.open("w", newline="", encoding="utf-8") as file:
        csv.writer(file).writerow(FIELDS)
print("Ready. Add candidate videos under training/data/videos/.")

### Annotation helpers

Collapsed implementation: pose tracking, rep detection, playback and resumable caching. No fatigue labels are inferred here.

In [ ]:
MODELS = {
    name: (
        f"https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
        f"pose_landmarker_{name}/float16/1/pose_landmarker_{name}.task"
    )
    for name in ("full", "heavy")
}
LANDMARKS = {
    "left": (11, 13, 15),
    "right": (12, 14, 16),
}


@dataclass
class Analysis:
    frames: pd.DataFrame
    reps: pd.DataFrame
    diagnostics: pd.DataFrame
    annotated_mp4: bytes
    preview_rgb: np.ndarray | None
    fps: float
    duration_s: float
    tracked_fraction: float
    left_tracked_fraction: float
    right_tracked_fraction: float
    model_variant: str
    source_path: Path
    source_sha256: str


def ensure_model(variant: str) -> Path:
    if variant not in MODELS:
        raise ValueError(f"Unknown pose model: {variant}")
    path = ROOT / "training" / "models" / f"pose_landmarker_{variant}.task"
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(".download")
        try:
            urlretrieve(MODELS[variant], temporary)
            temporary.replace(path)
        finally:
            temporary.unlink(missing_ok=True)
    return path


def _angle(shoulder: np.ndarray, elbow: np.ndarray, wrist: np.ndarray) -> float:
    a = shoulder - elbow
    b = wrist - elbow
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator < 1e-8:
        return np.nan
    cosine = np.clip(np.dot(a, b) / denominator, -1.0, 1.0)
    return float(np.degrees(np.arccos(cosine)))


def _landmarks(result, frame_rgb: np.ndarray) -> dict[str, float]:
    row = {key: np.nan for key in (
        "left_flexion", "right_flexion", "left_wrist_x", "left_wrist_y",
        "right_wrist_x", "right_wrist_y", "left_elbow_x", "left_elbow_y",
        "right_elbow_x", "right_elbow_y", "left_upper_arm", "right_upper_arm",
        "left_shoulder_x", "left_shoulder_y", "right_shoulder_x", "right_shoulder_y",
        "shoulder_width",
    )}
    if not result.pose_landmarks:
        return row

    points = result.pose_landmarks[0]
    shoulders = [points[11], points[12]]
    if min(p.visibility for p in shoulders) >= 0.6:
        row["shoulder_width"] = float(np.hypot(
            shoulders[0].x - shoulders[1].x,
            shoulders[0].y - shoulders[1].y,
        ))

    for side, indices in LANDMARKS.items():
        shoulder, elbow, wrist = (points[i] for i in indices)
        if min(p.visibility for p in (shoulder, elbow, wrist)) < 0.6:
            continue
        s = np.array([shoulder.x, shoulder.y])
        e = np.array([elbow.x, elbow.y])
        w = np.array([wrist.x, wrist.y])
        row[f"{side}_flexion"] = 180.0 - _angle(s, e, w)
        row[f"{side}_shoulder_x"] = float(s[0])
        row[f"{side}_shoulder_y"] = float(s[1])
        row[f"{side}_wrist_x"] = float(w[0] - s[0])
        row[f"{side}_wrist_y"] = float(w[1] - s[1])
        row[f"{side}_elbow_x"] = float(e[0] - s[0])
        row[f"{side}_elbow_y"] = float(e[1] - s[1])
        row[f"{side}_upper_arm"] = float(np.linalg.norm(e - s))
        height, width = frame_rgb.shape[:2]
        for point in (shoulder, elbow, wrist):
            cv2.circle(frame_rgb, (int(point.x * width), int(point.y * height)), 6, (255, 210, 0), -1)
        cv2.line(frame_rgb, (int(s[0] * width), int(s[1] * height)),
                 (int(e[0] * width), int(e[1] * height)), (80, 230, 255), 3)
        cv2.line(frame_rgb, (int(e[0] * width), int(e[1] * height)),
                 (int(w[0] * width), int(w[1] * height)), (80, 230, 255), 3)
    return row


def _smooth(values: pd.Series, fps: float) -> np.ndarray:
    missing_limit = max(1, int(round(0.2 * fps)))
    filled = values.interpolate(limit=missing_limit, limit_area="inside").to_numpy(dtype=float)
    window = max(5, int(round(0.25 * fps)) | 1)
    if len(filled) < window:
        return filled
    output = np.full_like(filled, np.nan)
    valid = np.isfinite(filled)
    edges = np.diff(np.r_[False, valid, False].astype(int))
    for start, end in zip(np.where(edges == 1)[0], np.where(edges == -1)[0]):
        segment = filled[start:end]
        if len(segment) >= window:
            output[start:end] = savgol_filter(segment, window, 2)
        else:
            output[start:end] = segment
    return output


def detect_reps(frames: pd.DataFrame, fps: float) -> pd.DataFrame:
    """Propose curl cycles from wrist lift; retain uncertainty for human review."""
    times = frames["time_s"].to_numpy(dtype=float)
    signals = {}
    coverage = {}
    for side in ("left", "right"):
        frames[f"{side}_flexion_smooth"] = _smooth(frames[f"{side}_flexion"], fps)
        length = frames[f"{side}_upper_arm"].to_numpy(dtype=float)
        usable_length = length[np.isfinite(length) & (length > 0.02)]
        scale = float(np.median(usable_length)) if len(usable_length) else np.nan
        lift = -frames[f"{side}_wrist_y"] / scale
        signals[side] = _smooth(lift, fps)
        coverage[side] = float(np.isfinite(signals[side]).mean())
        frames[f"{side}_lift_signal"] = signals[side]
    frames["mean_flexion"] = frames[["left_flexion_smooth", "right_flexion_smooth"]].mean(axis=1)
    primary = max(coverage, key=coverage.get)
    other = "right" if primary == "left" else "left"
    signal = signals[primary]
    valid = np.isfinite(signal)
    diagnostics = []
    reps = []
    columns = ["rep", "start_s", "top_s", "end_s", "review_warning"]
    if valid.sum() < 10:
        frames.attrs["diagnostics"] = pd.DataFrame([{"time_s": None, "result": "Insufficient visible wrist movement; manual annotation needed"}])
        return pd.DataFrame(columns=columns)
    low, high = np.nanpercentile(signal, [10, 90])
    span = high - low
    if span < 0.25:
        frames.attrs["diagnostics"] = pd.DataFrame([{"time_s": None, "result": "Insufficient wrist range; manual annotation needed"}])
        return pd.DataFrame(columns=columns)
    search = np.where(valid, signal, low - 10 * span)
    peaks, _ = find_peaks(search, prominence=max(0.18, 0.25 * span), distance=max(1, int(0.5 * fps)))
    for position, peak in enumerate(peaks):
        def note(message):
            diagnostics.append({"time_s": round(float(times[peak]), 2), "result": message})
        left_edge = 0 if position == 0 else int(peaks[position - 1])
        right_edge = len(signal) - 1 if position == len(peaks) - 1 else int(peaks[position + 1])
        before = signal[left_edge:peak + 1]
        after = signal[peak:right_edge + 1]
        if not np.isfinite(before).any() or not np.isfinite(after).any():
            note("Needs manual annotation: incomplete tracking around movement")
            continue
        tolerance = 0.06 * span
        before_bottom = np.nanmin(before)
        after_bottom = np.nanmin(after)
        starts = np.flatnonzero(np.isfinite(before) & (before <= before_bottom + tolerance))
        ends = np.flatnonzero(np.isfinite(after) & (after <= after_bottom + tolerance))
        start = left_edge + int(starts[-1])
        end = peak + int(ends[0])
        if not start < peak < end:
            note("Rejected: incomplete up-and-down cycle")
            continue
        if times[peak] - times[start] < 0.15 or times[end] - times[peak] < 0.15:
            note("Rejected: brief tracking spike")
            continue
        if min(signal[peak] - signal[start], signal[peak] - signal[end]) < max(0.25, 0.55 * span):
            note("Rejected: partial movement below the recording's typical range")
            continue
        if max(signal[start], signal[end]) > low + 0.3 * span:
            note("Rejected: no clear return to bottom")
            continue
        if np.isfinite(signal[start:end + 1]).mean() < 0.75:
            note("Needs manual annotation: primary wrist missing")
            continue
        bend = frames[f"{primary}_flexion_smooth"].iloc[start:end + 1].to_numpy(dtype=float)
        if np.isfinite(bend).sum() >= 5 and np.nanpercentile(bend, 90) - np.nanpercentile(bend, 10) < 25:
            note("Rejected: wrist motion without clear elbow flexion")
            continue
        warnings = []
        if times[end] - times[start] > 20:
            warnings.append("long cycle")
        other_segment = signals[other][start:end + 1]
        if np.isfinite(other_segment).mean() < 0.65:
            warnings.append(f"{other} arm partly occluded")
        else:
            other_peak = start + int(np.nanargmax(other_segment))
            if abs(times[other_peak] - times[peak]) > 0.6:
                warnings.append("check arm synchronization")
        if np.isfinite(signal[start:end + 1]).mean() < 0.95:
            warnings.append("short tracking gaps")
        reps.append({"rep": len(reps) + 1, "start_s": round(float(times[start]), 2),
                     "top_s": round(float(times[peak]), 2), "end_s": round(float(times[end]), 2),
                     "review_warning": "; ".join(warnings)})
        note(f"Suggested rep {len(reps)}" + (f" — {'; '.join(warnings)}" if warnings else ""))
    frames.attrs["diagnostics"] = pd.DataFrame(diagnostics, columns=["time_s", "result"])
    return pd.DataFrame(reps, columns=columns)


def _draw_overlay(frame_bgr: np.ndarray, row: pd.Series, count: int, phase: str) -> np.ndarray:
    height, width = frame_bgr.shape[:2]
    panel_height = min(48, height // 5)
    cv2.rectangle(frame_bgr, (0, 0), (width, panel_height), (20, 25, 35), -1)
    cv2.putText(frame_bgr, f"COUNT {count}", (12, 33), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)
    cv2.putText(frame_bgr, phase, (max(180, width // 3), 31), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (80, 220, 255), 2)
    cv2.rectangle(frame_bgr, (0, height - 42), (width, height), (20, 25, 35), -1)

    for side, color in (("left", (255, 180, 50)), ("right", (80, 230, 100))):
        keys = [f"{side}_shoulder_x", f"{side}_shoulder_y", f"{side}_elbow_x",
                f"{side}_elbow_y", f"{side}_wrist_x", f"{side}_wrist_y"]
        if not np.isfinite(row[keys].to_numpy(dtype=float)).all():
            continue
        shoulder = np.array([row[f"{side}_shoulder_x"], row[f"{side}_shoulder_y"]])
        elbow = shoulder + np.array([row[f"{side}_elbow_x"], row[f"{side}_elbow_y"]])
        wrist = shoulder + np.array([row[f"{side}_wrist_x"], row[f"{side}_wrist_y"]])
        pixels = [(int(p[0] * width), int(p[1] * height)) for p in (shoulder, elbow, wrist)]
        cv2.line(frame_bgr, pixels[0], pixels[1], color, 4)
        cv2.line(frame_bgr, pixels[1], pixels[2], color, 4)
        for label, point in zip(("S", "E", "W"), pixels):
            cv2.circle(frame_bgr, point, 7, color, -1)
            cv2.putText(frame_bgr, label, (point[0] + 8, point[1] - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        bend = row[f"{side}_flexion_smooth"]
        if np.isfinite(bend):
            x = 12 if side == "left" else width // 2
            cv2.putText(frame_bgr, f"{side.upper()} {bend:.0f} deg", (x, height - 12),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    return frame_bgr


def _report_progress(phase: str, current: int, total: int, started: float, last: float, *, finished: bool = False) -> float:
    now = time.monotonic()
    if not finished and now - last < 10:
        return last
    elapsed = now - started
    fraction = min(1.0, current / total) if total > 0 else None
    position = f"{current:,}/{total:,} frames ({fraction:.0%})" if fraction is not None else f"{current:,} frames"
    remaining = elapsed * (total - current) / current if total > current > 0 else 0
    estimate = f" | about {remaining / 60:.1f} min remaining" if remaining else ""
    print(f"  {phase}: {position} | {elapsed / 60:.1f} min elapsed{estimate}", flush=True)
    return now


def _annotated_video(video_path: str | Path, frames: pd.DataFrame, reps: pd.DataFrame, fps: float) -> bytes:
    """Make a browser-playable video showing landmarks, phase, and running count."""
    capture = cv2.VideoCapture(str(video_path))
    started = last_progress = time.monotonic()
    print(f"  Creating annotated playback: {len(frames):,} frames", flush=True)
    starts = reps["start_s"].to_numpy(dtype=float) if not reps.empty else np.array([])
    tops = reps["top_s"].to_numpy(dtype=float) if not reps.empty else np.array([])
    ends = reps["end_s"].to_numpy(dtype=float) if not reps.empty else np.array([])
    try:
        with TemporaryDirectory() as folder:
            folder = Path(folder)
            raw_path = folder / "overlay_raw.mp4"
            output_path = folder / "overlay.mp4"
            writer = None
            try:
                for index, row in frames.iterrows():
                    ok, frame = capture.read()
                    if not ok:
                        break
                    height, width = frame.shape[:2]
                    if width > 960:
                        frame = cv2.resize(frame, (960, round(height * 960 / width)))
                    if writer is None:
                        height, width = frame.shape[:2]
                        writer = cv2.VideoWriter(str(raw_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
                        if not writer.isOpened():
                            raise RuntimeError("Could not create the annotated video.")
                    time_s = float(row["time_s"])
                    count = int(np.searchsorted(ends, time_s, side="right"))
                    phase = "WAITING FOR CURL"
                    if not np.isfinite(row[["left_flexion", "right_flexion"]].to_numpy(dtype=float)).all():
                        phase = "ARMS NOT TRACKED"
                    for start, top, end in zip(starts, tops, ends):
                        if start <= time_s < end:
                            phase = "LIFTING" if time_s < top else "LOWERING"
                            break
                    writer.write(_draw_overlay(frame, row, count, phase))
                    last_progress = _report_progress("Playback", index + 1, len(frames), started, last_progress)
            finally:
                if writer is not None:
                    writer.release()
            _report_progress("Playback", len(frames), len(frames), started, last_progress, finished=True)
            print("  Encoding browser playback…", flush=True)
            result = subprocess.run(
                ["ffmpeg", "-y", "-loglevel", "error", "-i", str(raw_path),
                 "-c:v", "libx264", "-preset", "veryfast", "-crf", "25",
                 "-pix_fmt", "yuv420p", "-movflags", "+faststart", "-an", str(output_path)],
                capture_output=True, text=True,
            )
            if result.returncode != 0:
                raise RuntimeError(f"Could not encode the annotated video: {result.stderr.strip()}")
            return output_path.read_bytes()
    finally:
        capture.release()


def file_sha256(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def discard_source_audio(path):
    probe = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "a", "-show_entries", "stream=index", "-of", "json", str(path)], check=True, capture_output=True, text=True)
    if not json.loads(probe.stdout).get("streams"):
        return
    old_hash = file_sha256(path)
    silent = path.with_name(path.stem + ".silent" + path.suffix)
    backup = ROOT / "dummy" / "backups" / "notebook_audio_removal" / old_hash
    backup.mkdir(parents=True, exist_ok=True)
    try:
        subprocess.run(["ffmpeg", "-nostdin", "-v", "error", "-copyts", "-i", str(path), "-map", "0:v:0", "-c:v", "copy", "-an", "-map_metadata", "0", "-avoid_negative_ts", "disabled", str(silent)], check=True, capture_output=True)
        shutil.copy2(path, backup / path.name)
        silent.replace(path)
        new_hash = file_sha256(path)
        for table in (DATA_DIR / "processed").glob("*.csv"):
            values = pd.read_csv(table, keep_default_na=False)
            if "video_sha256" in values and values.video_sha256.eq(old_hash).any():
                shutil.copy2(table, backup / table.name)
                values.loc[values.video_sha256.eq(old_hash), "video_sha256"] = new_hash
                values.to_csv(table, index=False)
        print(f"Removed audio: {path.name}", flush=True)
    finally:
        silent.unlink(missing_ok=True)


def analyze_video(video_path: str | Path, max_seconds: int | None = None, model_variant: str = "full") -> Analysis:
    video_path = Path(video_path).resolve()
    discard_source_audio(video_path)
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("FFmpeg is required for annotated playback. Install it before analysis.")
    source_sha256 = file_sha256(video_path)
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise ValueError("Could not open this video. Try an MP4 or MOV recorded on a phone.")
    fps = float(capture.get(cv2.CAP_PROP_FPS))
    if not 5 <= fps <= 120:
        capture.release()
        raise ValueError("The video frame rate could not be read reliably.")
    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    if max_seconds is not None:
        total_frames = min(total_frames, int(max_seconds * fps)) if total_frames > 0 else int(max_seconds * fps)
    started = last_progress = time.monotonic()
    print(f"  Tracking with {model_variant}: {total_frames:,} frames at {fps:.1f} FPS. Long videos can take several minutes.", flush=True)
    model_path = ensure_model(model_variant)
    options = mp.tasks.vision.PoseLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(model_path)),
        running_mode=mp.tasks.vision.RunningMode.VIDEO,
        num_poses=1,
    )
    rows = []
    preview = None
    max_frames = int(max_seconds * fps) if max_seconds is not None else None
    previous_timestamp_ms = -1
    try:
        with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
            index = 0
            while max_frames is None or index < max_frames:
                ok, frame_bgr = capture.read()
                if not ok:
                    break
                height, width = frame_bgr.shape[:2]
                if width > 960:
                    frame_bgr = cv2.resize(frame_bgr, (960, round(height * 960 / width)))
                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
                reported_ms = capture.get(cv2.CAP_PROP_POS_MSEC)
                fallback_ms = round(index * 1000 / fps)
                timestamp_ms = round(reported_ms) if np.isfinite(reported_ms) and reported_ms > previous_timestamp_ms else fallback_ms
                timestamp_ms = max(timestamp_ms, previous_timestamp_ms + 1)
                previous_timestamp_ms = timestamp_ms
                image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
                result = landmarker.detect_for_video(image, timestamp_ms)
                annotated = frame_rgb.copy()
                row = _landmarks(result, annotated)
                row["time_s"] = timestamp_ms / 1000
                rows.append(row)
                index += 1
                last_progress = _report_progress("Tracking", index, total_frames, started, last_progress)
                if preview is None and np.isfinite(row["left_flexion"]) and np.isfinite(row["right_flexion"]):
                    preview = annotated
    finally:
        capture.release()
    if not rows:
        raise ValueError("The video contains no readable frames.")
    _report_progress("Tracking", len(rows), total_frames, started, last_progress, finished=True)
    print("  Finding candidate repetitions…", flush=True)
    frames = pd.DataFrame(rows)
    reps = detect_reps(frames, fps)
    diagnostics = frames.attrs.get("diagnostics", pd.DataFrame())
    frames.attrs.clear()
    tracked = frames[["left_flexion", "right_flexion"]].notna().all(axis=1).mean()
    left_tracked = frames["left_flexion"].notna().mean()
    right_tracked = frames["right_flexion"].notna().mean()
    annotated_mp4 = _annotated_video(video_path, frames, reps, fps)
    return Analysis(frames, reps, diagnostics, annotated_mp4, preview, fps,
                    float(frames["time_s"].iloc[-1]), float(tracked),
                    float(left_tracked), float(right_tracked), model_variant, video_path, source_sha256)

In [ ]:
BOUNDARIES = ["start_s", "top_s", "end_s"]


def validate_boundaries(reps: pd.DataFrame, duration_s: float) -> pd.DataFrame:
    if not isinstance(reps, pd.DataFrame) or not set(BOUNDARIES).issubset(reps.columns):
        raise ValueError("Provide start_s, top_s and end_s for each completed rep.")
    result = reps[BOUNDARIES].copy()
    if result.empty:
        raise ValueError("No completed repetitions to review.")
    for column in BOUNDARIES:
        result[column] = pd.to_numeric(result[column], errors="raise")
    values = result.to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError("Rep timestamps must be finite numbers.")
    if (values < 0).any() or (values > duration_s + 0.02).any():
        raise ValueError("Rep timestamps must fall within the analyzed recording.")
    if not ((values[:, 0] < values[:, 1]) & (values[:, 1] < values[:, 2])).all():
        raise ValueError("Each rep must satisfy start_s < top_s < end_s.")
    if len(values) > 1 and (values[1:, 0] < values[:-1, 2] - 0.02).any():
        raise ValueError("Reps must be in chronological order and must not overlap.")
    result = result.reset_index(drop=True)
    result.insert(0, "rep", np.arange(1, len(result) + 1))
    return result


def reviewed_playback(analysis: Analysis, reps: pd.DataFrame) -> bytes:
    if file_sha256(analysis.source_path) != analysis.source_sha256:
        raise ValueError("The source video changed. Run analysis again.")
    boundaries = validate_boundaries(reps, analysis.duration_s)
    return _annotated_video(analysis.source_path, analysis.frames, boundaries, analysis.fps)


def export_annotations(
    analysis: Analysis,
    reps: pd.DataFrame,
    record: dict,
    set_id: str,
    output: Path,
    *,
    count_reviewed: bool = False,
    complete_uncut_set: bool = False,
    unsuccessful_next_rep: bool = False,
    endpoint_evidence: str = "",
    collector_confirmed_endpoint: bool = False,
    continuous_to_endpoint: bool = False,
    overwrite: bool = False,
) -> Path:
    confirmations = [count_reviewed, complete_uncut_set, unsuccessful_next_rep, collector_confirmed_endpoint, continuous_to_endpoint, overwrite]
    if any(type(value) is not bool for value in confirmations):
        raise ValueError("Review confirmations must be True or False.")
    if record.get("reuse_status") != "confirmed" or not record.get("reuse_evidence", "").strip():
        raise ValueError("Confirm reuse permission and document its evidence in the manifest first.")
    for field in ("video_id", "participant_id", "source_recording_id"):
        if not re.fullmatch(r"[A-Za-z0-9_-]+", record.get(field, "")):
            raise ValueError(f"Provide a code for {field} in the manifest.")
    if not re.fullmatch(r"S\d{2,}", set_id):
        raise ValueError("Use a set code such as S01.")
    if record.get("quality_status") == "unusable":
        raise ValueError("This recording is marked unusable in the manifest.")
    if file_sha256(analysis.source_path) != analysis.source_sha256:
        raise ValueError("The source video changed. Run analysis again.")
    result = validate_boundaries(reps, analysis.duration_s)
    visible_endpoint = complete_uncut_set and unsuccessful_next_rep
    reported_endpoint = collector_confirmed_endpoint and continuous_to_endpoint
    labeled = count_reviewed and (visible_endpoint or reported_endpoint)
    if labeled and not endpoint_evidence.strip():
        raise ValueError("Document the visible endpoint or collector confirmation supporting the label.")
    result.insert(0, "set_id", set_id)
    for field in ("source_recording_id", "participant_id", "video_id"):
        result.insert(0, field, record[field])
    result["relative_path"] = record["relative_path"]
    result["video_sha256"] = analysis.source_sha256
    result["annotation_pose_model"] = analysis.model_variant
    result["timestamp_basis"] = "OpenCV POS_MSEC with monotonic FPS fallback; inspect timing before training"
    result["count_reviewed"] = count_reviewed
    result["complete_uncut_set"] = complete_uncut_set
    result["unsuccessful_next_rep"] = unsuccessful_next_rep
    result["endpoint_evidence"] = endpoint_evidence.strip()
    result["continuous_to_endpoint"] = continuous_to_endpoint
    result["endpoint_source"] = "collector_confirmation" if reported_endpoint else "visible_attempt_review" if visible_endpoint else "unconfirmed"
    result["rir_label_status"] = ("collector_confirmed_failure" if reported_endpoint else "reviewer_confirmed_failure") if labeled else "unlabeled"
    result["reps_left"] = len(result) - result["rep"] if labeled else pd.NA
    output = Path(output)
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open("w" if overwrite else "x", newline="", encoding="utf-8") as file:
        result.to_csv(file, index=False)
    return output


def register_videos(videos: list[Path], video_dir: Path, manifest_path: Path) -> int:
    with manifest_path.open(newline="", encoding="utf-8-sig") as file:
        reader = csv.DictReader(file)
        fields = reader.fieldnames
        rows = list(reader)
    if not fields or not {"relative_path", "video_id"}.issubset(fields):
        raise ValueError("Manifest must have video_id and relative_path columns.")
    known = [row["relative_path"].strip() for row in rows]
    if len(set(known)) != len(known):
        raise ValueError("Resolve duplicate relative_path rows in the manifest first.")
    added = 0
    for video in videos:
        relative = video.resolve().relative_to(video_dir.resolve()).as_posix()
        if relative in known:
            continue
        row = dict.fromkeys(fields, "")
        row.update(video_id="V" + hashlib.sha256(relative.encode()).hexdigest()[:12], relative_path=relative)
        for field, value in {"reuse_status": "pending", "complete_uncut_set": "unknown", "label_status": "pending", "quality_status": "pending"}.items():
            if field in row:
                row[field] = value
        rows.append(row)
        known.append(relative)
        added += 1
    if added:
        temporary = manifest_path.with_suffix(".tmp")
        try:
            with temporary.open("w", newline="", encoding="utf-8") as file:
                writer = csv.DictWriter(file, fieldnames=fields)
                writer.writeheader()
                writer.writerows(rows)
            temporary.replace(manifest_path)
        finally:
            temporary.unlink(missing_ok=True)
    return added


def processor_fingerprint(model_variant: str) -> str:
    def describe(code):
        constants = [describe(value) if isinstance(value, CodeType) else repr(value) for value in code.co_consts]
        return [code.co_code.hex(), code.co_names, constants]

    functions = [ensure_model, _angle, _landmarks, _smooth, detect_reps,
                 _draw_overlay, _annotated_video, analyze_video, file_sha256]
    source = json.dumps([describe(function.__code__) for function in functions], sort_keys=True)
    values = [source, file_sha256(ensure_model(model_variant))]
    values.extend(version(package) for package in ["mediapipe", "opencv-contrib-python", "numpy", "pandas", "scipy"])
    return hashlib.sha256("".join(values).encode()).hexdigest()


def _valid_cache(folder: Path, source_hash: str, processor: str) -> dict | None:
    try:
        metadata = json.loads((folder / "analysis.json").read_text())
        if metadata["source_sha256"] != source_hash or metadata["processor_sha256"] != processor:
            return None
        expected = {"suggested.mp4", "frames.csv", "suggested_reps.csv", "diagnostics.csv"}
        if set(metadata["files"]) != expected:
            return None
        for name, digest in metadata["files"].items():
            if file_sha256(folder / name) != digest:
                return None
        return metadata
    except (OSError, ValueError, KeyError, TypeError):
        return None


def batch_analyze(videos: list[Path], video_dir: Path, output_dir: Path, model_variant: str = "heavy", force: bool = False) -> pd.DataFrame:
    if not videos:
        return pd.DataFrame(columns=["relative_path", "status", "suggested_reps", "both_arms_tracked", "cache_dir", "error"])
    processor = processor_fingerprint(model_variant)
    results = []
    for index, source in enumerate(videos, start=1):
        row = {"relative_path": source.relative_to(video_dir).as_posix(), "status": "error", "suggested_reps": None, "both_arms_tracked": None, "cache_dir": "", "error": ""}
        print(f"[{index}/{len(videos)}] {row['relative_path']}", flush=True)
        try:
            source_hash = file_sha256(source)
            folder = output_dir / f"{source_hash}_{model_variant}_{processor[:12]}"
            metadata = None if force else _valid_cache(folder, source_hash, processor)
            if metadata is None:
                analysis = analyze_video(source, model_variant=model_variant)
                if analysis.source_sha256 != source_hash or file_sha256(source) != source_hash:
                    raise ValueError("Source changed during analysis. Run again.")
                folder.mkdir(parents=True, exist_ok=True)
                (folder / "suggested.mp4").write_bytes(analysis.annotated_mp4)
                analysis.frames.to_csv(folder / "frames.csv", index=False)
                analysis.reps.reindex(columns=["rep", "start_s", "top_s", "end_s", "review_warning"]).to_csv(folder / "suggested_reps.csv", index=False)
                analysis.diagnostics.reindex(columns=["time_s", "result"]).to_csv(folder / "diagnostics.csv", index=False)
                metadata = {key: getattr(analysis, key) for key in ["fps", "duration_s", "tracked_fraction", "left_tracked_fraction", "right_tracked_fraction", "model_variant", "source_sha256"]}
                metadata.update(processor_sha256=processor, suggested_reps=len(analysis.reps), files={name: file_sha256(folder / name) for name in ["suggested.mp4", "frames.csv", "suggested_reps.csv", "diagnostics.csv"]})
                temporary = folder / "analysis.json.tmp"
                temporary.write_text(json.dumps(metadata, indent=2) + "\n")
                temporary.replace(folder / "analysis.json")
                row["status"] = "processed"
            else:
                row["status"] = "cached"
            row.update(suggested_reps=metadata["suggested_reps"], both_arms_tracked=metadata["tracked_fraction"], cache_dir=str(folder.resolve()))
            print(f"  {row['status']}: {row['suggested_reps']} suggested reps", flush=True)
        except Exception as error:
            row["error"] = str(error)
            print(f"  failed: {error}", flush=True)
        results.append(row)
        output_dir.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(results).to_csv(output_dir / "batch_summary.csv", index=False)
    return pd.DataFrame(results)


def load_analysis(source: Path, folder: Path, model_variant: str) -> Analysis:
    source = source.resolve()
    metadata = _valid_cache(folder, file_sha256(source), processor_fingerprint(model_variant))
    if metadata is None or metadata["model_variant"] != model_variant:
        raise ValueError("Cache missing, changed or outdated. Run the batch again.")
    return Analysis(
        frames=pd.read_csv(folder / "frames.csv"), reps=pd.read_csv(folder / "suggested_reps.csv"),
        diagnostics=pd.read_csv(folder / "diagnostics.csv"), annotated_mp4=(folder / "suggested.mp4").read_bytes(),
        preview_rgb=None, fps=metadata["fps"], duration_s=metadata["duration_s"],
        tracked_fraction=metadata["tracked_fraction"], left_tracked_fraction=metadata["left_tracked_fraction"],
        right_tracked_fraction=metadata["right_tracked_fraction"], model_variant=metadata["model_variant"],
        source_path=source, source_sha256=metadata["source_sha256"],
    )

def update_reviewed_dataset(annotation_path: Path, record: dict) -> Path:
    incoming = pd.read_csv(annotation_path)
    output = DATA_DIR / "processed" / "reviewed_reps.csv"
    existing = pd.read_csv(output) if output.exists() else pd.DataFrame()
    identity = incoming[["video_id", "set_id"]].drop_duplicates()
    if len(identity) != 1:
        raise ValueError("One annotation export must contain exactly one video/set.")
    video_id, set_id = identity.iloc[0]
    if not existing.empty:
        existing = existing.loc[~((existing.video_id == video_id) & (existing.set_id == set_id))].copy()
    eligible = (incoming.rir_label_status.isin(["collector_confirmed_failure", "reviewer_confirmed_failure"])
                & incoming.reps_left.notna())
    if eligible.all() and record.get("reuse_status") == "confirmed":
        incoming["reuse_evidence"] = record["reuse_evidence"]
        combined = pd.concat([existing, incoming], ignore_index=True)
    else:
        combined = existing
    if combined.empty:
        raise ValueError("No confirmed training labels remain; review the saved annotation export.")
    if combined.duplicated(["participant_id", "set_id", "rep"]).any():
        raise ValueError("Participant/set/rep codes conflict. Give each person's sets distinct IDs.")
    output.parent.mkdir(parents=True, exist_ok=True)
    if output.exists():
        shutil.copy2(output, output.with_suffix(".previous.csv"))
    temporary = output.with_suffix(".tmp")
    combined.to_csv(temporary, index=False)
    temporary.replace(output)
    return output




def export_candidate_dataset(summary: pd.DataFrame) -> Path:
    inventory = pd.read_csv(MANIFEST_PATH, dtype=str).fillna("")
    if set(summary.relative_path) != set(inventory.relative_path):
        raise ValueError("The batch summary does not cover the current inventory. Finish processing all clips first.")
    tables = []
    for _, item in summary.iterrows():
        if item["status"] not in {"processed", "cached"}:
            continue
        matches = inventory[inventory.relative_path == item.relative_path]
        if len(matches) != 1:
            raise ValueError("Each processed clip needs one inventory entry.")
        record = matches.iloc[0]
        folder = Path(item.cache_dir)
        metadata = json.loads((folder / "analysis.json").read_text())
        if metadata["source_sha256"] != file_sha256(VIDEO_DIR / item.relative_path):
            raise ValueError(f"Source changed after processing: {item.relative_path}")
        proposed = pd.read_csv(folder / "suggested_reps.csv")
        if proposed.empty:
            continue
        result = validate_boundaries(proposed, metadata["duration_s"])
        result["video_id"] = record.video_id
        result["participant_id"] = record.participant_id
        result["set_id"] = record.get("set_id", "") or "S01"
        result["relative_path"] = record.relative_path
        result["video_sha256"] = metadata["source_sha256"]
        result["reps_left"] = pd.NA
        result["rir_label_status"] = "unlabeled"
        result["endpoint_source"] = "unconfirmed"
        result["endpoint_evidence"] = record.endpoint_evidence
        result["review_method"] = "automatic_mediapipe"
        result["boundary_status"] = record.get("rep_boundary_status", "") or "automatic_suggestion"
        result["reuse_status"] = record.reuse_status
        tables.append(result)
    columns = ["rep", "start_s", "top_s", "end_s", "video_id", "participant_id", "set_id", "relative_path",
               "video_sha256", "reps_left", "rir_label_status", "endpoint_source", "endpoint_evidence",
               "review_method", "boundary_status", "reuse_status"]
    combined = pd.concat(tables, ignore_index=True) if tables else pd.DataFrame(columns=columns)
    output = DATA_DIR / "processed" / "candidate_reps.csv"
    output.parent.mkdir(parents=True, exist_ok=True)
    temporary = output.with_suffix(".tmp")
    combined.to_csv(temporary, index=False)
    temporary.replace(output)
    print(f"Saved {len(combined)} automatic candidate reps. RIR targets remain blank until supported.")
    return output




## 2. Process all videos

Set `RUN_BATCH = True` and run the cell below. Wait for processing to finish before reviewing. Unchanged completed videos are skipped.

This also saves automatic candidate boundaries for notebook 02. Those rows have no RIR targets; they are separate from confirmed training annotations.

In [ ]:
POSE_MODEL = "heavy"
RUN_BATCH = True
FORCE_REPROCESS = False
batch_summary = None
analysis = None
reviewed_reps = None
review_path = None
review_record = None

if RUN_BATCH:
    videos = sorted((path for path in VIDEO_DIR.rglob("*")
                    if path.is_file() and path.suffix.lower() in EXTENSIONS),
                    key=lambda path: [int(part) if part.isdigit() else part.lower()
                                      for part in re.split(r"(\d+)", path.relative_to(VIDEO_DIR).as_posix())])
    for path in videos:
        discard_source_audio(path)
    added = register_videos(videos, VIDEO_DIR, MANIFEST_PATH)
    print(f"Files: {len(videos)} | New inventory entries: {added}")
    print("Processing started. Progress appears below; wait for the result table.", flush=True)
    batch_summary = batch_analyze(videos, VIDEO_DIR, REVIEW_DIR, POSE_MODEL, force=FORCE_REPROCESS)
    batch_summary.to_csv(REVIEW_DIR / "batch_summary.csv", index=False)
    summary = batch_summary.drop(columns="cache_dir").copy()
    summary.index = range(1, len(summary) + 1)
    summary.index.name = "review_index"
    display(summary)
    export_candidate_dataset(batch_summary)
else:
    print("Set RUN_BATCH = True to process the folder.")

## 3. Watch and review

Use the video picker or **Previous/Next** buttons. Check suggested counts and flagged reps against the original recording. Suggestions use wrist lift with elbow-flexion checks; occluded or asynchronous arms are flagged instead of silently discarding the whole rep. Switching videos resets unsaved boundary edits.

In [ ]:
analysis = None
reviewed_reps = None
review_path = None
review_record = None
REVIEW_INDEX = None

if globals().get("review_panel") is not None:
    review_panel.close()
if globals().get("batch_summary") is None:
    saved_summary = REVIEW_DIR / "batch_summary.csv"
    if not saved_summary.exists():
        raise RuntimeError("Run Section 2 with RUN_BATCH = True first.")
    batch_summary = pd.read_csv(saved_summary).fillna("")

successful = batch_summary.index[batch_summary["status"].isin(["processed", "cached"])].tolist()
if not successful:
    display(batch_summary[["relative_path", "error"]])
    print("No successful results. Check batch processing above.")
else:
    options = [(f"{i + 1}. {row['relative_path']} — {int(row['suggested_reps'])} suggested reps", i + 1)
               for i, row in batch_summary.iterrows() if i in successful]
    video_picker = widgets.Dropdown(options=options, description="Video:", layout=widgets.Layout(width="95%"))
    previous_button = widgets.Button(description="Previous")
    next_button = widgets.Button(description="Next")
    review_output = widgets.Output()

    def open_review(index):
        global analysis, reviewed_reps, review_path, review_record, REVIEW_INDEX
        analysis = reviewed_reps = review_path = review_record = None
        REVIEW_INDEX = index
        position = [value for _, value in options].index(index)
        previous_button.disabled = position == 0
        next_button.disabled = position == len(options) - 1
        review_output.clear_output(wait=True)
        with review_output:
            try:
                row = batch_summary.iloc[index - 1]
                source = VIDEO_DIR / row["relative_path"]
                folder = Path(row["cache_dir"])
                loaded = load_analysis(source, folder, POSE_MODEL)
                with MANIFEST_PATH.open(newline="", encoding="utf-8-sig") as file:
                    matches = [record for record in csv.DictReader(file) if record["relative_path"].strip() == row["relative_path"]]
                if len(matches) != 1:
                    raise ValueError("Expected one inventory entry for this video.")
                review_record = {key: (value or "").strip() for key, value in matches[0].items()}
                analysis, review_path = loaded, folder
                reviewed_reps = analysis.reps[BOUNDARIES].copy()
                print("Video ID:", review_record["video_id"])
                print("Participant:", review_record["participant_id"] or "Not set")
                print("Source group:", review_record["source_recording_id"] or "Not set")
                print(f"Suggested reps: {len(reviewed_reps)} | Both arms tracked: {analysis.tracked_fraction:.0%}")
                display(Video(str(folder / "suggested.mp4"), embed=True, width=420))
                display(analysis.reps.fillna(""))
                for text in analysis.reps.get("review_warning", pd.Series(dtype=str)).dropna().unique():
                    if text:
                        print("Review:", text)
                diagnostics_panel = widgets.Output()
                with diagnostics_panel:
                    display(analysis.diagnostics)
                details = widgets.Accordion(children=[diagnostics_panel], titles=["Detection diagnostics"])
                display(details)
                columns = [column for column in ["left_lift_signal", "right_lift_signal"] if column in analysis.frames]
                if columns:
                    axes = analysis.frames.plot(x="time_s", y=columns, figsize=(12, 3))
                    for end in analysis.reps["end_s"]:
                        axes.axvline(end, color="gray", alpha=0.25)
                    axes.set(xlabel="Seconds", ylabel="Relative wrist height", xlim=(0, analysis.duration_s))
                    plt.show()
                    plt.close(axes.figure)
            except Exception as error:
                analysis = reviewed_reps = review_path = review_record = None
                print(f"Could not open review: {error}")
                print("Rerun batch processing to refresh missing or outdated results.")

    def select_video(change):
        if change["name"] == "value":
            open_review(change["new"])

    def step_video(offset):
        values = [value for _, value in options]
        position = values.index(video_picker.value) + offset
        if 0 <= position < len(values):
            video_picker.value = values[position]

    video_picker.observe(select_video, names="value")
    previous_button.on_click(lambda _: step_video(-1))
    next_button.on_click(lambda _: step_video(1))
    review_panel = widgets.VBox([video_picker, widgets.HBox([previous_button, next_button]), review_output])
    display(review_panel)
    open_review(video_picker.value)

### Correct boundaries if needed

Edit `reviewed_reps` below. Times are seconds: `start_s` → `top_s` → `end_s`. Drop extra rows or add missed rows, then replay. Corrections do not require rerunning MediaPipe.

In [ ]:
# Optional corrections, using zero-based row indices:
# reviewed_reps.loc[0, "start_s"] = corrected_time
# reviewed_reps = reviewed_reps.drop(index=extra_row).reset_index(drop=True)
# reviewed_reps.loc[len(reviewed_reps)] = [start_seconds, top_seconds, end_seconds]

if reviewed_reps is not None:
    reviewed_reps = reviewed_reps.sort_values("start_s").reset_index(drop=True)
    display(reviewed_reps)

In [ ]:
if analysis is None or reviewed_reps is None:
    print("Open a batch result first.")
elif reviewed_reps.empty:
    print("No completed reps. Add reviewed boundaries or try the other pose model.")
else:
    checked = validate_boundaries(reviewed_reps, analysis.duration_s)
    playback = review_path / "corrected.mp4"
    playback.write_bytes(reviewed_playback(analysis, checked))
    display(Video(str(playback), embed=True, width=420))
    print(f"Corrected reps: {len(checked)}")

## 4. Save the review

Enter details here; the inventory updates automatically. Copy the current video ID into `REVIEWED_VIDEO_ID`. Empty identity/evidence fields reuse saved values.

RIR labels require a reviewed count and documented endpoint. Confirm either a visible unsuccessful next attempt in a complete set, or the collector's confirmation of inability to perform another full rep with uninterrupted footage from the labeled reps to that endpoint. A recording may begin mid-set; its initial partial rep is excluded. Collector-confirmed labels are marked separately from visible-attempt reviews.

A video ending or shaking alone does not establish failure. Timing uses OpenCV timestamps with an FPS fallback; verify it before motion-frequency analysis.

In [ ]:
SAVE_REVIEW = False
REVIEWED_VIDEO_ID = ""
PARTICIPANT_ID = ""
SOURCE_RECORDING_ID = ""
SET_ID = ""
REUSE_CONFIRMED = False
REUSE_EVIDENCE = ""
COUNT_REVIEWED = False
COMPLETE_UNCUT_SET = False
UNSUCCESSFUL_NEXT_REP = False
COLLECTOR_CONFIRMED_ENDPOINT = False
CONTINUOUS_TO_ENDPOINT = False
ENDPOINT_EVIDENCE = ""
REVIEW_NOTES = ""
OVERWRITE_EXPORT = False

if SAVE_REVIEW:
    if analysis is None or reviewed_reps is None or review_record is None:
        raise ValueError("Open and review a batch result first.")
    if REVIEWED_VIDEO_ID != review_record["video_id"]:
        raise ValueError("Copy the current video ID into REVIEWED_VIDEO_ID before saving.")
    if type(REVIEW_INDEX) is not int or batch_summary is None or not 1 <= REVIEW_INDEX <= len(batch_summary):
        raise ValueError("Choose a valid review index.")
    current = VIDEO_DIR / batch_summary.iloc[REVIEW_INDEX - 1]["relative_path"]
    if current.resolve() != analysis.source_path or POSE_MODEL != analysis.model_variant:
        raise ValueError("Selection changed. Open and review the selected result again.")
    if type(REUSE_CONFIRMED) is not bool:
        raise ValueError("REUSE_CONFIRMED must be True or False.")
    record = review_record.copy()
    record["participant_id"] = PARTICIPANT_ID.strip() or record["participant_id"]
    record["source_recording_id"] = SOURCE_RECORDING_ID.strip() or record["source_recording_id"]
    record["reuse_evidence"] = REUSE_EVIDENCE.strip() or record["reuse_evidence"]
    if REUSE_CONFIRMED:
        record["reuse_status"] = "confirmed"
    endpoint_evidence = ENDPOINT_EVIDENCE.strip() or record["endpoint_evidence"]
    SET_ID = SET_ID.strip() or record.get("set_id", "").strip() or "S01"
    annotation_path = review_path / f"{record['video_id']}_{SET_ID}_annotations.csv"
    export_annotations(
        analysis, reviewed_reps, record, SET_ID, annotation_path,
        count_reviewed=COUNT_REVIEWED, complete_uncut_set=COMPLETE_UNCUT_SET,
        unsuccessful_next_rep=UNSUCCESSFUL_NEXT_REP, endpoint_evidence=endpoint_evidence,
        collector_confirmed_endpoint=COLLECTOR_CONFIRMED_ENDPOINT, continuous_to_endpoint=CONTINUOUS_TO_ENDPOINT,
        overwrite=OVERWRITE_EXPORT,
    )
    verified = COUNT_REVIEWED and ((COMPLETE_UNCUT_SET and UNSUCCESSFUL_NEXT_REP) or (COLLECTOR_CONFIRMED_ENDPOINT and CONTINUOUS_TO_ENDPOINT))
    record["label_status"] = ("collector_confirmed_failure" if COLLECTOR_CONFIRMED_ENDPOINT else "supported") if verified else "pending"
    record["endpoint_evidence"] = endpoint_evidence
    record["complete_uncut_set"] = "yes" if COMPLETE_UNCUT_SET else record["complete_uncut_set"]
    if UNSUCCESSFUL_NEXT_REP:
        record["stop_reason"] = "unsuccessful_next_rep"
    if REVIEW_NOTES.strip():
        record["review_notes"] = REVIEW_NOTES.strip()
    with MANIFEST_PATH.open(newline="", encoding="utf-8-sig") as file:
        reader = csv.DictReader(file)
        fields, records = reader.fieldnames, list(reader)
    matching = [i for i, existing in enumerate(records) if existing["relative_path"].strip() == record["relative_path"]]
    if len(matching) != 1:
        raise ValueError("Inventory changed. Annotations were saved; reopen the review before updating the inventory.")
    records[matching[0]] = record
    temporary = MANIFEST_PATH.with_suffix(".tmp")
    try:
        with temporary.open("w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=fields)
            writer.writeheader()
            writer.writerows(records)
        temporary.replace(MANIFEST_PATH)
    finally:
        temporary.unlink(missing_ok=True)
    review_record = record
    combined_path = update_reviewed_dataset(annotation_path, record)
    print("Updated training labels:", combined_path.relative_to(ROOT))
    print("Saved:", annotation_path.relative_to(ROOT))
    print("RIR labels:", "reviewer-confirmed" if verified else "blank — endpoint/count not fully confirmed")
else:
    print("Complete the review fields and set SAVE_REVIEW = True to save.")

## Output

Each processed recording has an annotated replay, suggested boundaries and diagnostics. Saved annotations contain original-video references, reviewed timestamps, participant/source codes and label evidence—no pose, speed or instability features. Use reviewed labels with original video frames in the training notebook. Keep participant and source groups separate across evaluation splits.

## Saved visual audit

Additional-video observations and exclusions are stored separately from confirmed training labels. These sampled-frame reviews do not automatically certify failure endpoints.

In [ ]:
audit_path = REVIEW_DIR / "visual_audit" / "clip_review.csv"
if audit_path.exists():
    display(pd.read_csv(audit_path)[["video_id", "exercise_observed", "review_status", "exact_rir_eligible", "observations"]])
else:
    print("No saved visual audit yet.")